# EDA, Clean และ Data Prep: ระดับน้ำทะเลรายอำเภอ (2020–2025)

ลำดับการทำงาน: โหลดข้อมูล → EDA ก่อน Clean → Clean → Data Prep → EDA หลังเตรียมข้อมูล → Save CSV

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

filename = 'NST_district_monthly_sea_level_2020_2025.csv'
root = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / 'raw_data' / filename).is_file()), None)
if root is None:
    raise FileNotFoundError(f'ไม่พบไฟล์ raw_data/{filename}')
source_path = root / 'raw_data' / filename
years = [2020, 2021, 2022, 2023, 2024, 2025]
all_data = pd.read_csv(source_path)
raw_data = all_data.loc[all_data['year'].isin(years)].copy()
print(f'ไฟล์ต้นทาง: {source_path}')
print(f'ข้อมูลทั้งหมด {len(all_data)} แถว | ข้อมูลปี {years}: {len(raw_data)} แถว')
display(raw_data.head())

## 2. EDA ก่อน Clean: ตรวจข้อมูลและแสดงกราฟ

สำรวจข้อมูลระดับน้ำทะเลจาก CMEMS และ HYCOM แยกกัน โดยคงค่าว่างตามข้อมูลต้นฉบับ และแสดงแนวโน้มค่าเฉลี่ยรายปี

In [ ]:
sea_level_columns = [
    'ssh_cmems_mean_m', 'ssh_cmems_max_m', 'ssh_cmems_min_m',
    'ssh_hycom_mean_m', 'ssh_hycom_max_m', 'ssh_hycom_min_m',
]
print('จำนวนแถวและคอลัมน์:', raw_data.shape)
print('ค่าว่างต่อคอลัมน์:\n', raw_data.isna().sum().to_string())
print('แถวซ้ำทั้งหมด:', int(raw_data.duplicated().sum()))
print('ข้อมูลซ้ำของอำเภอ ปี และเดือน:', int(raw_data.duplicated(['district_code', 'year', 'month']).sum()))
print('จำนวนเดือนต่อปี:\n', raw_data.groupby('year')['month'].nunique().to_string())
print('จำนวนอำเภอต่อปี:\n', raw_data.groupby('year')['district'].nunique().to_string())
display(raw_data[sea_level_columns].describe().round(3))

raw_yearly = raw_data.groupby('year', as_index=False).agg(
    cmems_mean_m=('ssh_cmems_mean_m', 'mean'),
    hycom_mean_m=('ssh_hycom_mean_m', 'mean'),
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(raw_yearly['year'], raw_yearly['cmems_mean_m'], marker='o', linewidth=2.5)
axes[0].set(title='Raw data: annual mean sea level (CMEMS)', xlabel='Year', ylabel='Mean sea level (source unit)', xticks=years)
axes[0].grid(axis='y', alpha=.3)
axes[1].plot(raw_yearly['year'], raw_yearly['hycom_mean_m'], marker='o', linewidth=2.5, color='#D97706')
axes[1].set(title='Raw data: annual mean sea level (HYCOM)', xlabel='Year', ylabel='Mean sea level (source unit)', xticks=years)
axes[1].grid(axis='y', alpha=.3)
display(fig)
plt.close(fig)
display(raw_yearly.round(3))

In [ ]:
raw_cmems_district_year = raw_data.groupby(['district', 'year'])['ssh_cmems_mean_m'].mean().unstack('year').reindex(columns=years)
raw_hycom_district_year = raw_data.groupby(['district', 'year'])['ssh_hycom_mean_m'].mean().unstack('year').reindex(columns=years)
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
raw_cmems_district_year.plot(kind='bar', ax=axes[0], width=.8)
axes[0].set(title='Raw data: annual mean sea level by district (CMEMS)', xlabel='District', ylabel='Mean sea level (source unit)')
raw_hycom_district_year.plot(kind='bar', ax=axes[1], width=.8)
axes[1].set(title='Raw data: annual mean sea level by district (HYCOM)', xlabel='District', ylabel='Mean sea level (source unit)')
for ax in axes:
    ax.legend(title='Year')
    ax.tick_params(axis='x', labelrotation=55)
    ax.grid(axis='y', alpha=.25)
display(fig)
plt.close(fig)

## 3. Clean ข้อมูล

จัดรูปแบบชื่อจังหวัดและอำเภอ แปลงชนิดข้อมูล ลบแถวซ้ำโดยใช้คีย์อำเภอ-ปี-เดือน ตรวจค่าระดับน้ำที่มีข้อมูล และคงค่าว่างจากต้นทางไว้

In [ ]:
clean_data = raw_data.copy()
for column in ['province', 'district', 'month_th']:
    clean_data[column] = clean_data[column].astype('string').str.strip()
clean_data['district'] = clean_data['district'].str.replace(r'\s+', ' ', regex=True)
clean_data['year'] = pd.to_numeric(clean_data['year'], errors='raise').astype('int64')
clean_data['month'] = pd.to_numeric(clean_data['month'], errors='raise').astype('int64')
clean_data['district_code'] = pd.to_numeric(clean_data['district_code'], errors='raise').astype('int64')
for column in sea_level_columns:
    clean_data[column] = pd.to_numeric(clean_data[column], errors='coerce')
rows_before = len(clean_data)
clean_data = clean_data.drop_duplicates(subset=['district_code', 'year', 'month']).copy()
print(f'ลบแถวซ้ำของอำเภอ-ปี-เดือน: {rows_before - len(clean_data)} แถว')
if not clean_data['year'].isin(years).all() or not clean_data['month'].between(1, 12).all():
    raise ValueError('พบปีหรือเดือนไม่อยู่ในช่วงที่ร้องขอ')
required_columns = ['province', 'district', 'district_code', 'year', 'month', 'month_th']
if clean_data[required_columns].isna().any().any():
    raise ValueError('พบค่าว่างในคอลัมน์สำคัญ กรุณาตรวจสอบข้อมูลต้นฉบับ')
if clean_data[['province', 'district', 'month_th']].eq('').any().any():
    raise ValueError('พบชื่อจังหวัด อำเภอ หรือเดือนว่าง')
for prefix in ['ssh_cmems', 'ssh_hycom']:
    value_columns = [f'{prefix}_{stat}_m' for stat in ['mean', 'max', 'min']]
    has_values = clean_data[value_columns].notna().any(axis=1)
    complete_values = clean_data[value_columns].notna().all(axis=1)
    if (has_values & ~complete_values).any():
        raise ValueError(f'พบชุดค่าระดับน้ำ {prefix} ที่มีข้อมูลไม่ครบ')
    valid = clean_data.loc[complete_values]
    if not valid[f'{prefix}_min_m'].le(valid[f'{prefix}_mean_m']).all() or not valid[f'{prefix}_mean_m'].le(valid[f'{prefix}_max_m']).all():
        raise ValueError(f'ค่าต่ำสุด ค่าเฉลี่ย และค่าสูงสุดของ {prefix} ไม่สอดคล้องกัน')
print(f'Clean เสร็จ: {len(clean_data)} แถว | ค่าว่างของระดับน้ำคงไว้ตามข้อมูลต้นฉบับ')
print('ค่าว่างในคอลัมน์ระดับน้ำ:\n', clean_data[sea_level_columns].isna().sum().to_string())

## 4. Data Prep

เตรียมตารางวิเคราะห์รายเดือน เพิ่มวันที่เริ่มต้นเดือน จัดชนิดข้อมูล เรียงตามอำเภอ ปี และเดือน พร้อมคงค่าว่างจากต้นทาง

In [ ]:
data_prep = clean_data.copy()
data_prep['year'] = data_prep['year'].astype('int64')
data_prep['month'] = data_prep['month'].astype('int64')
data_prep['month_start'] = pd.to_datetime(
    dict(year=data_prep['year'], month=data_prep['month'], day=1),
    errors='raise',
)
prepared_columns = [
    'province', 'district', 'district_code', 'year', 'month', 'month_th', 'month_start',
    *sea_level_columns,
]
data_prep = data_prep[prepared_columns].sort_values(
    ['district_code', 'year', 'month']
).reset_index(drop=True)
if not data_prep['year'].isin(years).all():
    raise ValueError('พบปีนอกช่วงที่ร้องขอ')
if data_prep.duplicated(['district_code', 'year', 'month']).any():
    raise ValueError('พบข้อมูลซ้ำของอำเภอ ปี และเดือน')
if data_prep[['province', 'district', 'district_code', 'year', 'month', 'month_th', 'month_start']].isna().any().any():
    raise ValueError('พบค่าว่างในคอลัมน์สำคัญหลัง Data Prep')
print('ขนาดตาราง Data Prep:', data_prep.shape)
print('ปี:', sorted(data_prep['year'].unique().tolist()))
print('จำนวนอำเภอ:', data_prep['district'].nunique())
print('ค่าว่างของระดับน้ำที่คงไว้:\n', data_prep[sea_level_columns].isna().sum().to_string())
display(data_prep.head(10))

## 5. EDA หลัง Clean และ Data Prep

ตรวจแนวโน้มค่าเฉลี่ยระดับน้ำรายปีและค่าเฉลี่ยรายอำเภอจาก `data_prep` หลังผ่าน Clean และ Data Prep แล้ว โดยแสดง CMEMS และ HYCOM แยกกัน

In [ ]:
prepared_yearly = data_prep.groupby('year', as_index=False).agg(
    cmems_mean_m=('ssh_cmems_mean_m', 'mean'),
    hycom_mean_m=('ssh_hycom_mean_m', 'mean'),
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(prepared_yearly['year'], prepared_yearly['cmems_mean_m'], marker='o', linewidth=2.5)
axes[0].set(title='Prepared data: annual mean sea level (CMEMS)', xlabel='Year', ylabel='Mean sea level (source unit)', xticks=years)
axes[0].grid(axis='y', alpha=.3)
axes[1].plot(prepared_yearly['year'], prepared_yearly['hycom_mean_m'], marker='o', linewidth=2.5, color='#D97706')
axes[1].set(title='Prepared data: annual mean sea level (HYCOM)', xlabel='Year', ylabel='Mean sea level (source unit)', xticks=years)
axes[1].grid(axis='y', alpha=.3)
display(fig)
plt.close(fig)
display(prepared_yearly.round(3))

prepared_cmems_district_year = data_prep.groupby(['district', 'year'])['ssh_cmems_mean_m'].mean().unstack('year').reindex(columns=years)
prepared_hycom_district_year = data_prep.groupby(['district', 'year'])['ssh_hycom_mean_m'].mean().unstack('year').reindex(columns=years)
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
prepared_cmems_district_year.plot(kind='bar', ax=axes[0], width=.8)
axes[0].set(title='Prepared data: annual mean sea level by district (CMEMS)', xlabel='District', ylabel='Mean sea level (source unit)')
prepared_hycom_district_year.plot(kind='bar', ax=axes[1], width=.8)
axes[1].set(title='Prepared data: annual mean sea level by district (HYCOM)', xlabel='District', ylabel='Mean sea level (source unit)')
for ax in axes:
    ax.legend(title='Year')
    ax.tick_params(axis='x', labelrotation=55)
    ax.grid(axis='y', alpha=.25)
display(fig)
plt.close(fig)

## 6. Save CSV

บันทึก `data_prep` หลัง Clean และ Data Prep ลงใน `clean_data_set/` ขั้นตอนบันทึกไฟล์แยกเป็นเซลล์สุดท้าย

In [ ]:
output_dir = root / 'clean_data_set'
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / 'NST_district_monthly_sea_level_2020_2025_clean.csv'
data_prep.to_csv(csv_path, index=False, encoding='utf-8-sig', float_format='%.6f', date_format='%Y-%m-%d')
print(f'บันทึก CSV แล้ว: {csv_path}')
print(f'{len(data_prep)} แถว | {data_prep["district"].nunique()} อำเภอ | ปี {years}')